# Cab Cancellation Prediction - EDA and Model Training

This notebook explores the synthetic cab-booking dataset, checks for missing values and duplicates, visualizes key patterns, and trains a basic machine learning model.

## Project goal
Predict whether a cab booking is likely to be cancelled.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')

DATA_PATH = Path('../data/cab_bookings.csv')
df = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully')
print(df.head())
print('Shape:', df.shape)
print('Columns:', list(df.columns))

In [ ]:
# Display basic dataset information
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

The dataset appears clean, with no missing values or duplicate rows. We now inspect the distribution of the target variable and key numeric features.

In [ ]:
# Statistical summary
print(df.describe().T)

# Target distribution
sns.countplot(data=df, x='cancellation')
plt.title('Cancellation Target Distribution')
plt.xlabel('Cancellation')
plt.ylabel('Count')
plt.show()

print('Cancellation rate:', df['cancellation'].mean())

In [ ]:
# Numeric feature distributions
numeric_cols = ['distance_km', 'estimated_fare', 'driver_rating', 'customer_rating', 'driver_acceptance_rate', 'surge_multiplier', 'estimated_pickup_time_minutes']
for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[col], bins=25, kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

In [ ]:
# Categorical feature analysis
cat_cols = ['vehicle_type', 'weather', 'traffic_condition', 'payment_method', 'booking_source']
for col in cat_cols:
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f'Counts by {col}')
    plt.xticks(rotation=30)
    plt.show()

In [ ]:
# Correlation analysis
numeric_df = df.select_dtypes(include=[np.number])
correlation = numeric_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation, cmap='coolwarm', annot=False)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Cancellation rate by category
for col in ['vehicle_type', 'weather', 'traffic_condition', 'payment_method', 'booking_hour', 'surge_multiplier']:
    if col == 'booking_hour':
        grouped = df.groupby(col)['cancellation'].mean().sort_index()
    elif col == 'surge_multiplier':
        grouped = df.groupby(pd.cut(df[col], bins=[1, 1.5, 2, 2.5, 3.5], include_lowest=True))['cancellation'].mean()
    else:
        grouped = df.groupby(col)['cancellation'].mean().sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    grouped.plot(kind='bar')
    plt.title(f'Cancellation Rate by {col}')
    plt.ylabel('Cancellation Rate')
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

A few important patterns appear: cancellations increase when traffic is heavy, the weather is bad, pickup time is long, or the surge rate is higher. These are exactly the kinds of relationships we want the model to learn.

In [ ]:
# Train a simple model as a demonstration
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

feature_columns = [
    'distance_km', 'estimated_fare', 'booking_hour', 'booking_day', 'booking_day_of_week',
    'passenger_count', 'driver_rating', 'customer_rating', 'driver_experience_years',
    'driver_acceptance_rate', 'customer_previous_cancellations', 'customer_total_bookings',
    'driver_previous_cancellations', 'surge_multiplier', 'estimated_pickup_time_minutes',
    'weather', 'traffic_condition', 'vehicle_type', 'payment_method', 'booking_source', 'is_weekend'
]

X = df[feature_columns]
y = df['cancellation']

numeric_features = [
    'distance_km', 'estimated_fare', 'booking_hour', 'booking_day', 'booking_day_of_week',
    'passenger_count', 'driver_rating', 'customer_rating', 'driver_experience_years',
    'driver_acceptance_rate', 'customer_previous_cancellations', 'customer_total_bookings',
    'driver_previous_cancellations', 'surge_multiplier', 'estimated_pickup_time_minutes', 'is_weekend'
]
categorical_features = ['weather', 'traffic_condition', 'vehicle_type', 'payment_method', 'booking_source']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('F1-score:', f1_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

## Summary
This notebook shows the key data patterns, confirms that the target is learnable, and demonstrates a basic baseline model. The project then compares several models and saves the best one for deployment.